# Demand Forecasting - Self-Contained Project Notebook

## Objective
Build a standalone demand forecasting workflow that runs with local data when available and falls back to realistic synthetic data when not.

## Expected input schema
`date`, `store`, `item`, `sales`

## Output artifacts
- `output/metrics_summary.csv`
- `output/validation_forecasts.csv`

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error

SEED = 42
rng = np.random.default_rng(SEED)

project_dir = Path.cwd()
if not (project_dir / "Demand_Forecasting.ipynb").exists():
    project_dir = Path.cwd() / "HandsOn-Projects" / "Demand_Forecasting_Project"

data_dir = project_dir / "data"
output_dir = project_dir / "output"
data_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

DATA_PATH = data_dir / "train.csv"

def make_synthetic_demand() -> pd.DataFrame:
    dates = pd.date_range("2021-01-01", "2023-12-31", freq="D")
    stores = np.arange(1, 6)
    items = np.arange(1, 11)
    rows = []
    for store in stores:
        store_effect = 1.0 + store * 0.08
        for item in items:
            item_effect = 1.0 + item * 0.03
            base = 30 + 2 * item + 3 * store
            for d in dates:
                dow = d.dayofweek
                month = d.month
                weekly = 6 if dow in [4, 5] else 0
                yearly = 10 * np.sin((month / 12) * 2 * np.pi)
                noise = rng.normal(0, 3)
                sales = max(0, base * store_effect * item_effect + weekly + yearly + noise)
                rows.append((d, store, item, round(sales, 0)))
    return pd.DataFrame(rows, columns=["date", "store", "item", "sales"])

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH, parse_dates=["date"])
else:
    df = make_synthetic_demand()
    df.to_csv(DATA_PATH, index=False)

df = df.sort_values(["store", "item", "date"]).reset_index(drop=True)
print("Project dir:", project_dir)
print("Data path:", DATA_PATH)
print("Rows:", len(df))
df.head()

## 1) Data prep and temporal split

### Why this matters
Time-aware splits prevent leakage and simulate production forecasting behavior.

This section:
- Validates schema
- Creates calendar features and lag/rolling features
- Splits by date into train/validation

In [ ]:
required_cols = {"date", "store", "item", "sales"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df["date"] = pd.to_datetime(df["date"])
df["sales"] = pd.to_numeric(df["sales"], errors="coerce")
df = df.dropna(subset=["sales"]).copy()

df["dow"] = df["date"].dt.dayofweek
df["month"] = df["date"].dt.month
df["is_weekend"] = (df["dow"] >= 5).astype(int)

df = df.sort_values(["store", "item", "date"]).reset_index(drop=True)
grp = df.groupby(["store", "item"])
df["lag_1"] = grp["sales"].shift(1)
df["lag_7"] = grp["sales"].shift(7)
df["roll_mean_7"] = df.groupby(["store", "item"])["sales"].transform(lambda s: s.shift(1).rolling(7, min_periods=7).mean())
df["roll_std_7"] = df.groupby(["store", "item"])["sales"].transform(lambda s: s.shift(1).rolling(7, min_periods=7).std())

df_feat = df.dropna().copy()
split_date = df_feat["date"].quantile(0.8)
train = df_feat[df_feat["date"] <= split_date].copy()
valid = df_feat[df_feat["date"] > split_date].copy()

print("Split date:", split_date)
print("Train rows:", len(train), "Valid rows:", len(valid))

## 2) Baseline models

We evaluate two fast baselines:
- Naive forecast (`lag_1`)
- 7-day rolling mean forecast (`roll_mean_7`)

In [ ]:
def evaluate(y_true: pd.Series, y_pred: pd.Series, label: str) -> dict:
    mae = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mape = float((np.abs((y_true - y_pred) / np.clip(y_true, 1e-6, None))).mean() * 100)
    wape = float(np.abs(y_true - y_pred).sum() / np.clip(np.abs(y_true).sum(), 1e-6, None) * 100)
    return {"model": label, "MAE": mae, "RMSE": rmse, "MAPE_pct": mape, "WAPE_pct": wape}

valid_eval = valid[["date", "store", "item", "sales", "lag_1", "roll_mean_7"]].copy()
valid_eval["pred_naive_lag1"] = valid_eval["lag_1"]
valid_eval["pred_roll7"] = valid_eval["roll_mean_7"]

metrics = [
    evaluate(valid_eval["sales"], valid_eval["pred_naive_lag1"], "naive_lag1"),
    evaluate(valid_eval["sales"], valid_eval["pred_roll7"], "rolling_mean_7"),
]
metrics_df = pd.DataFrame(metrics).sort_values("WAPE_pct")
metrics_df

## 3) Lightweight global model

A compact linear model is trained on lag + calendar features to provide a stronger baseline while keeping dependencies minimal.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

feature_cols_num = ["lag_1", "lag_7", "roll_mean_7", "roll_std_7"]
feature_cols_cat = ["store", "item", "dow", "month", "is_weekend"]

X_train = train[feature_cols_num + feature_cols_cat]
y_train = train["sales"]
X_valid = valid[feature_cols_num + feature_cols_cat]
y_valid = valid["sales"]

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), feature_cols_num),
        ("cat", OneHotEncoder(handle_unknown="ignore"), feature_cols_cat),
    ]
)

model = Pipeline([
    ("prep", preprocess),
    ("ridge", Ridge(alpha=1.0, random_state=SEED)),
])
model.fit(X_train, y_train)
valid_eval["pred_ridge"] = model.predict(X_valid)

metrics.append(evaluate(y_valid, valid_eval["pred_ridge"], "ridge_global"))
metrics_df = pd.DataFrame(metrics).sort_values("WAPE_pct")
metrics_df

## 4) Persist artifacts and quick business interpretation

This section saves model outputs so the folder can be promoted to a standalone repository with deterministic outputs.

In [ ]:
best_model = metrics_df.iloc[0]["model"]
forecast_col = {
    "naive_lag1": "pred_naive_lag1",
    "rolling_mean_7": "pred_roll7",
    "ridge_global": "pred_ridge",
}[best_model]

to_save = valid_eval[["date", "store", "item", "sales", forecast_col]].rename(columns={forecast_col: "forecast"})
to_save.to_csv(output_dir / "validation_forecasts.csv", index=False)
metrics_df.to_csv(output_dir / "metrics_summary.csv", index=False)

print("Best model:", best_model)
print("Saved:", output_dir / "validation_forecasts.csv")
print("Saved:", output_dir / "metrics_summary.csv")
metrics_df